 ### Data loading and initial inspection

This analysis uses three files: `customers.csv`, `articles.csv`, and `transactions_train.csv`. Initially, I loaded five rows from each file to inspect their structure and confirm that they could be accessed correctly. I did not load the full transactions file because it contains more than 31 million rows and could exceed the available memory. Each row in `transactions_train.csv` represents one article purchased by one customer on a specific date through a sales channel; it does not necessarily represent a complete order. The `price` column contains normalized numerical values, but its currency and scaling method are not documented. Therefore, it will be used only for relative comparisons and will not be interpreted as revenue in euros.


In [2]:
import pandas as pd
import numpy as np
import pathlib as path

In [3]:
data_dir = path.Path.home() / "Downloads" / "hm_data"

In [4]:
data_dir.exists()

True

In [5]:
list(data_dir.iterdir())

[PosixPath('/Users/anastasiia/Downloads/hm_data/customers.csv'),
 PosixPath('/Users/anastasiia/Downloads/hm_data/customers.csv.zip'),
 PosixPath('/Users/anastasiia/Downloads/hm_data/.DS_Store'),
 PosixPath('/Users/anastasiia/Downloads/hm_data/hm_retention.duckdb'),
 PosixPath('/Users/anastasiia/Downloads/hm_data/articles.csv'),
 PosixPath('/Users/anastasiia/Downloads/hm_data/articles.csv.zip'),
 PosixPath('/Users/anastasiia/Downloads/hm_data/transactions_train.csv'),
 PosixPath('/Users/anastasiia/Downloads/hm_data/transactions_train.csv.zip')]

In [6]:
customers_sample = pd.read_csv(data_dir/"customers.csv", nrows=5)

In [7]:
customers_sample.head()

,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,NaN,NaN,ACTIVE,NONE,49,52043ee2162cf5aa7ee79974281641c6f11a68d276429a...
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,NaN,NaN,ACTIVE,NONE,25,2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93...
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,NaN,NaN,ACTIVE,NONE,24,64f17e6a330a85798e4998f62d0930d14db8db1c054af6...
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,NaN,NaN,ACTIVE,NONE,54,5d36574f52495e81f019b680c843c443bd343d5ca5b1c2...
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,1.0,1.0,ACTIVE,Regularly,52,25fa5ddee9aac01b35208d01736e57942317d756b32ddd...


In [8]:
articles_sample = pd.read_csv(data_dir/"articles.csv", nrows=5)

In [9]:
articles_sample.head()

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,108775015,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,9,Black,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
1,108775044,108775,Strap top,253,Vest top,Garment Upper body,1010016,Solid,10,White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
2,108775051,108775,Strap top (1),253,Vest top,Garment Upper body,1010017,Stripe,11,Off White,...,Jersey Basic,A,Ladieswear,1,Ladieswear,16,Womens Everyday Basics,1002,Jersey Basic,Jersey top with narrow shoulder straps.
3,110065001,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,9,Black,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."
4,110065002,110065,OP T-shirt (Idro),306,Bra,Underwear,1010016,Solid,10,White,...,Clean Lingerie,B,Lingeries/Tights,1,Ladieswear,61,Womens Lingerie,1017,"Under-, Nightwear","Microfibre T-shirt bra with underwired, moulde..."


In [10]:
transactions_sample = pd.read_csv(data_dir/"transactions_train.csv",nrows=5)

In [11]:
transactions_sample.head()

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.016932,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.016932,2


In [12]:
transactions_sample.nunique()

t_dat               1
customer_id         2
article_id          5
price               4
sales_channel_id    1
dtype: int64

In [13]:
purchase_ocassions_sample = transactions_sample[["t_dat","customer_id","sales_channel_id"]].drop_duplicates()

In [14]:
purchase_ocassions_sample.shape

(2, 3)

In [15]:
transactions_sample.dtypes

t_dat                   str
customer_id             str
article_id            int64
price               float64
sales_channel_id      int64
dtype: object

In [16]:
transactions_sample["t_dat"] = pd.to_datetime(transactions_sample["t_dat"])

In [17]:
transactions_sample.dtypes

t_dat               datetime64[us]
customer_id                    str
article_id                   int64
price                      float64
sales_channel_id             int64
dtype: object

In [18]:
transactions_path = data_dir / "transactions_train.csv"

In [19]:
transactions_path.exists()

True

In [20]:
transactions_path.stat().st_size

3488002253

# Convert the transactions file size from bytes to gigabytes

In [21]:
transactions_size_gb= transactions_path.stat().st_size/ 1024 ** 3

In [22]:
round(transactions_size_gb, 2)

3.25

In [23]:
import duckdb
duckdb.__version__

'1.5.5'

## DuckDB connection

DuckDB is used to query the complete transactions file efficiently without loading the entire CSV into pandas memory.

In [24]:
db_path = data_dir / "hm_retention.duckdb"

In [25]:
con = duckdb.connect(str(db_path))

In [26]:
con.sql("SELECT 42 AS connection_test").show()

┌─────────────────┐
│ connection_test │
│      int32      │
├─────────────────┤
│              42 │
└─────────────────┘



In [27]:
transactions_relation = con.read_csv(str(transactions_path))

In [28]:
transactions_relation.create_view("transactions", replace=True)

┌────────────┬──────────────────────────────────────────────────────────────────┬────────────┬───────────────────────┬──────────────────┐
│   t_dat    │                           customer_id                            │ article_id │         price         │ sales_channel_id │
│    date    │                             varchar                              │  varchar   │        double         │      int64       │
├────────────┼──────────────────────────────────────────────────────────────────┼────────────┼───────────────────────┼──────────────────┤
│ 2018-09-20 │ 000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318 │ 0663713001 │  0.050830508474576264 │                2 │
│ 2018-09-20 │ 000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318 │ 0541518023 │   0.03049152542372881 │                2 │
│ 2018-09-20 │ 00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2 │ 0505221004 │   0.01523728813559322 │                2 │
│ 2018-09-20 │ 00007d2de826758b65a

## Transaction sample audit

1. Each row represents one article purchased by a customer, not a complete order.
2. Five rows represent five articles, but they do not necessarily represent five orders. The sample contains two approximate purchase occasions, although the exact number of orders is unknown because there is no order ID.
3. The `price` column has a numeric `float64` data type. However, its currency and scale are not documented, so it cannot be interpreted as a price in euros.

## Full transactions audit with SQL

DuckDB is used to audit the complete transactions dataset without loading all rows into pandas memory.

In [29]:
con.sql("""
    SELECT *
    FROM transactions
    LIMIT 5
""").show()

┌────────────┬──────────────────────────────────────────────────────────────────┬────────────┬──────────────────────┬──────────────────┐
│   t_dat    │                           customer_id                            │ article_id │        price         │ sales_channel_id │
│    date    │                             varchar                              │  varchar   │        double        │      int64       │
├────────────┼──────────────────────────────────────────────────────────────────┼────────────┼──────────────────────┼──────────────────┤
│ 2018-09-20 │ 000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318 │ 0663713001 │ 0.050830508474576264 │                2 │
│ 2018-09-20 │ 000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318 │ 0541518023 │  0.03049152542372881 │                2 │
│ 2018-09-20 │ 00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2 │ 0505221004 │  0.01523728813559322 │                2 │
│ 2018-09-20 │ 00007d2de826758b65a93dd24c

In [30]:
con.sql("""
    DESCRIBE transactions
""").show()

┌──────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name    │ column_type │  null   │   key   │ default │  extra  │
│     varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ t_dat            │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ customer_id      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ article_id       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ price            │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ sales_channel_id │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
└──────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



In [31]:
con.sql("""
    SELECT
        COUNT(*) AS transactions_rows,
        COUNT(DISTINCT customer_id) AS unique_customer,
        COUNT(DISTINCT article_id) AS unique_article
    FROM transactions
""").show()

┌───────────────────┬─────────────────┬────────────────┐
│ transactions_rows │ unique_customer │ unique_article │
│       int64       │      int64      │     int64      │
├───────────────────┼─────────────────┼────────────────┤
│          31788324 │         1362281 │         104547 │
└───────────────────┴─────────────────┴────────────────┘



In [32]:
con.sql("""
    SELECT
        MIN(t_dat) AS start_date,
        MAX(t_dat) AS end_date,
        COUNT(DISTINCT t_dat) AS active_days
    FROM transactions
""").show()

┌────────────┬────────────┬─────────────┐
│ start_date │  end_date  │ active_days │
│    date    │    date    │    int64    │
├────────────┼────────────┼─────────────┤
│ 2018-09-20 │ 2020-09-22 │         734 │
└────────────┴────────────┴─────────────┘



In [33]:
con.sql("""
    SELECT
        COUNT(*) - COUNT(t_dat) AS missing_t_dat,
        COUNT(*) - COUNT(customer_id) AS missing_customer_id,
        COUNT(*) - COUNT(article_id) AS missing_article_id,
        COUNT(*) - COUNT(price) AS missing_price,
        COUNT(*) - COUNT(sales_channel_id) AS missing_sales_channel
    FROM transactions
""").show()

┌───────────────┬─────────────────────┬────────────────────┬───────────────┬───────────────────────┐
│ missing_t_dat │ missing_customer_id │ missing_article_id │ missing_price │ missing_sales_channel │
│     int64     │        int64        │       int64        │     int64     │         int64         │
├───────────────┼─────────────────────┼────────────────────┼───────────────┼───────────────────────┤
│             0 │                   0 │                  0 │             0 │                     0 │
└───────────────┴─────────────────────┴────────────────────┴───────────────┴───────────────────────┘



### Missing values finding

No missing values were found in any column of the transactions dataset. Therefore, no missing-value treatment is required before the retention analysis.

In [34]:
con.sql("""
    SELECT
        sales_channel_id,
        COUNT(*) AS transactions_rows
    FROM transactions
    GROUP BY sales_channel_id
    ORDER BY sales_channel_id
""").show()
        

┌──────────────────┬───────────────────┐
│ sales_channel_id │ transactions_rows │
│      int64       │       int64       │
├──────────────────┼───────────────────┤
│                1 │           9408462 │
│                2 │          22379862 │
└──────────────────┴───────────────────┘



In [35]:
con.sql("""
    SELECT
        sales_channel_id,
        COUNT(*) AS transaction_rows,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            2
        ) AS transaction_share_pct
    FROM transactions
    GROUP BY sales_channel_id
    ORDER BY sales_channel_id
""").show()

┌──────────────────┬──────────────────┬───────────────────────┐
│ sales_channel_id │ transaction_rows │ transaction_share_pct │
│      int64       │      int64       │        double         │
├──────────────────┼──────────────────┼───────────────────────┤
│                1 │          9408462 │                  29.6 │
│                2 │         22379862 │                  70.4 │
└──────────────────┴──────────────────┴───────────────────────┘



### Sales channel finding

Channel 2 accounts for approximately 70.4% of all transaction rows, while Channel 1 accounts for 29.6%. These percentages represent transaction lines rather than customers or complete orders. Because the channel labels are not documented, the analysis retains the original Channel 1 and Channel 2 names.

In [36]:
con.sql("""
    SELECT
        COUNT(*) AS approximate_purchase_occasions
    FROM (
        SELECT DISTINCT
            customer_id,
            t_dat,
            sales_channel_id
        FROM transactions
    ) AS purchase_occasions
""").show()

┌────────────────────────────────┐
│ approximate_purchase_occasions │
│             int64              │
├────────────────────────────────┤
│                        9174457 │
└────────────────────────────────┘



### Approximate purchase occasions

The 31,788,324 transaction rows were reduced to 9,174,457 approximate purchase occasions using each unique combination of customer, date, and sales channel. These occasions are not confirmed orders because the dataset does not contain an order ID.

In [37]:
con.sql("""
    SELECT
        customer_id,
        t_dat,
        sales_channel_id,
        COUNT(*) transaction_line
    FROM transactions
    GROUP BY
          customer_id,
          t_dat,
          sales_channel_id
    ORDER BY transaction_line DESC
    LIMIT 10
""").show()

┌──────────────────────────────────────────────────────────────────┬────────────┬──────────────────┬──────────────────┐
│                           customer_id                            │   t_dat    │ sales_channel_id │ transaction_line │
│                             varchar                              │    date    │      int64       │      int64       │
├──────────────────────────────────────────────────────────────────┼────────────┼──────────────────┼──────────────────┤
│ d00063b94dcb1342869d4994844a2742b5d62927f36843164fb3f818f630bca9 │ 2018-10-14 │                1 │              570 │
│ c2f0cdda2dc3042ccd9fcd8253fd8e368769840581e40aab1d87a64ff39987e3 │ 2018-12-17 │                2 │              336 │
│ 62fd7d41b587c72a95c31eca5046019ce4e802853397ffd00f354c17007ebd0b │ 2019-01-13 │                2 │              221 │
│ 246734d8f4a4252fcd5c7aa525055a2804b9a6fb3d4210e771a33ed98f2bce77 │ 2018-10-15 │                2 │              220 │
│ 94665b46e194622ccdbcadc0170f13a2f8ede1

In [38]:
con.sql("""
    SELECT
        article_id,
        price,
        COUNT(*) AS total_rows,
        COUNT(*) AS repeated_rows,
        COUNT(DISTINCT article_id) AS unique_articles
    FROM transactions
    WHERE customer_id == 'd00063b94dcb1342869d4994844a2742b5d62927f36843164fb3f818f630bca9'
        AND t_dat = '2018-10-14'
        AND sales_channel_id = 1
    GROUP BY article_id,
             price
    ORDER BY total_rows
""").show()

┌────────────┬──────────────────────┬────────────┬───────────────┬─────────────────┐
│ article_id │        price         │ total_rows │ repeated_rows │ unique_articles │
│  varchar   │        double        │   int64    │     int64     │      int64      │
├────────────┼──────────────────────┼────────────┼───────────────┼─────────────────┤
│ 0678342001 │ 0.008694915254237287 │          1 │             1 │               1 │
│ 0678342001 │  0.00676271186440678 │        569 │           569 │               1 │
└────────────┴──────────────────────┴────────────┴───────────────┴─────────────────┘



### Repeated transaction lines

An extreme customer-date-channel combination contained 570 transaction rows but only one unique article. Of these rows, 569 had the same article and price. The dataset does not provide an order ID or quantity field, so these rows cannot be reliably classified as duplicated records or multiple units. Therefore, repeated rows will not be removed automatically. The main retention metric is based on distinct customer purchase dates, regardless of sales channel. Multiple transaction lines or purchases through both channels on the same date count as one observed purchase date. Transaction-line counts and price-based measures require cautious interpretation.

In [39]:
con.sql("""
    SELECT
        customer_id,
        t_dat,
        sales_channel_id,
        COUNT(*) transaction_lines,
        COUNT(DISTINCT article_id) AS unique_articles
    FROM transactions
    GROUP BY
          customer_id,
          t_dat,
          sales_channel_id
    ORDER BY transaction_lines DESC
    LIMIT 10 
""").show()

┌──────────────────────────────────────────────────────────────────┬────────────┬──────────────────┬───────────────────┬─────────────────┐
│                           customer_id                            │   t_dat    │ sales_channel_id │ transaction_lines │ unique_articles │
│                             varchar                              │    date    │      int64       │       int64       │      int64      │
├──────────────────────────────────────────────────────────────────┼────────────┼──────────────────┼───────────────────┼─────────────────┤
│ d00063b94dcb1342869d4994844a2742b5d62927f36843164fb3f818f630bca9 │ 2018-10-14 │                1 │               570 │               1 │
│ c2f0cdda2dc3042ccd9fcd8253fd8e368769840581e40aab1d87a64ff39987e3 │ 2018-12-17 │                2 │               336 │              45 │
│ 62fd7d41b587c72a95c31eca5046019ce4e802853397ffd00f354c17007ebd0b │ 2019-01-13 │                2 │               221 │              28 │
│ 246734d8f4a4252fcd5c7aa52

In [40]:
con.execute("""
    CREATE OR REPLACE TABLE purchase_occasions AS
    SELECT
        customer_id,
        t_dat,
        sales_channel_id,
        COUNT(*) transaction_lines,
        COUNT(DISTINCT article_id) AS unique_articles
    FROM transactions
    GROUP BY
          customer_id,
          t_dat,
          sales_channel_id
""")
    

In [41]:
con.sql("""
    SELECT
        COUNT(*) AS total_occasions
    FROM purchase_occasions
""").show()

┌─────────────────┐
│ total_occasions │
│      int64      │
├─────────────────┤
│         9174457 │
└─────────────────┘



In [42]:
con.sql("""
    SELECT
        ROUND(AVG(transaction_lines),2) AS avg_transactions_line,
        MEDIAN(transaction_lines) AS median_transactions_line,
        MAX(transaction_lines) AS max_transactions_line,
        ROUND(AVG(unique_articles),2) AS avg_unique_articles,
        MEDIAN(unique_articles) AS median_unique_articles,
        MAX(unique_articles) AS MAX_unique_articles
    FROM purchase_occasions
""").show()

┌───────────────────────┬──────────────────────────┬───────────────────────┬─────────────────────┬────────────────────────┬─────────────────────┐
│ avg_transactions_line │ median_transactions_line │ max_transactions_line │ avg_unique_articles │ median_unique_articles │ MAX_unique_articles │
│        double         │          double          │         int64         │       double        │         double         │        int64        │
├───────────────────────┼──────────────────────────┼───────────────────────┼─────────────────────┼────────────────────────┼─────────────────────┤
│                  3.46 │                      2.0 │                   570 │                3.12 │                    2.0 │                 159 │
└───────────────────────┴──────────────────────────┴───────────────────────┴─────────────────────┴────────────────────────┴─────────────────────┘



##  Purchase occasion size

A typical purchase occasion contains two unique articles. The average is higher than the median because a small number of extreme observations influence the mean. Therefore, the maximum values should not be used to describe typical customer behavior.


In [43]:
con.sql("""
    SELECT
        QUANTILE_CONT(transaction_lines, 0.95),
        QUANTILE_CONT(transaction_lines, 0.99),
        QUANTILE_CONT(unique_articles, 0.95),
        QUANTILE_CONT(unique_articles, 0.99)
    FROM purchase_occasions
""").show()

┌────────────────────────────────────────┬────────────────────────────────────────┬──────────────────────────────────────┬──────────────────────────────────────┐
│ quantile_cont(transaction_lines, 0.95) │ quantile_cont(transaction_lines, 0.99) │ quantile_cont(unique_articles, 0.95) │ quantile_cont(unique_articles, 0.99) │
│                 double                 │                 double                 │                double                │                double                │
├────────────────────────────────────────┼────────────────────────────────────────┼──────────────────────────────────────┼──────────────────────────────────────┤
│                                   10.0 │                                   18.0 │                                  9.0 │                                 15.0 │
└────────────────────────────────────────┴────────────────────────────────────────┴──────────────────────────────────────┴──────────────────────────────────────┘



In [44]:
con.sql("""
    SELECT
        customer_id,
        MIN(t_dat) AS first_observed_purchase_date
    FROM purchase_occasions
    GROUP BY customer_id
    LIMIT 10
""").show()

┌──────────────────────────────────────────────────────────────────┬──────────────────────────────┐
│                           customer_id                            │ first_observed_purchase_date │
│                             varchar                              │             date             │
├──────────────────────────────────────────────────────────────────┼──────────────────────────────┤
│ fc139755810ae81829aca0ad83e787b2496551dc699387c43b837c1485e92546 │ 2018-11-30                   │
│ 39aaca995351934f732f04e438813d0bb2bf05b060a4cb187e59c8a88681bc75 │ 2018-10-22                   │
│ eda7087ec1378e69e9fa10038836c41ca9ee7686a9ae6e2b2d34fcbd4eea2cab │ 2018-10-19                   │
│ 5b14979ef1e5815ff394a538b1fba8fc964f746ef47b531cb251fe558e94de1c │ 2020-01-19                   │
│ 8173427ceea0d06b83e8600ed91b8dd1068f344e611e66c628b75e45c1eb29b7 │ 2019-06-27                   │
│ 936200e143b6cfb6139d4629309ea5f2f7a12213ea4060b7629a7a21488b19e6 │ 2019-04-03                   │


In [45]:
con.execute("""
    CREATE OR REPLACE TABLE customer_purchase_dates AS 
    SELECT DISTINCT
        customer_id,
        t_dat
    FROM purchase_occasions
""")
        

In [46]:
con.sql("""
    SELECT
        COUNT(*) AS total_customer_dates,
        COUNT(DISTINCT customer_id) AS unique_customers,
        COUNT(DISTINCT t_dat) AS calendar_dates
    FROM customer_purchase_dates
""").show()

┌──────────────────────┬──────────────────┬────────────────┐
│ total_customer_dates │ unique_customers │ calendar_dates │
│        int64         │      int64       │     int64      │
├──────────────────────┼──────────────────┼────────────────┤
│              9080179 │          1362281 │            734 │
└──────────────────────┴──────────────────┴────────────────┘



In [47]:
con.execute("""
CREATE OR REPLACE TABLE customer_lifecycle AS
WITH  sequenced_purchase AS (
    SELECT
        customer_id,
        t_dat,
        ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY t_dat) AS purchase_number,
        LEAD(t_dat) OVER(PARTITION BY customer_id ORDER BY t_dat) AS next_purchase_date
    FROM customer_purchase_dates
   
)
SELECT
    customer_id,
    t_dat AS first_date,
    next_purchase_date AS second_date,
    DATE_DIFF('day', t_dat, next_purchase_date) AS days_to_second,
    CASE
      WHEN t_dat <= DATE '2020-06-24' THEN 1
    ELSE 0
    END AS eligible_90d,
    CASE
    WHEN t_dat > DATE '2020-06-24' THEN NULL
    WHEN next_purchase_date IS NOT NULL
         AND DATE_DIFF('day', t_dat, next_purchase_date) <= 90 THEN 1
    ELSE 0
    END AS ret_90d
FROM sequenced_purchase
WHERE purchase_number = 1
""")

In [48]:
con.sql("""
    SELECT
        COUNT(*) AS total_customers,
        COUNT(DISTINCT customer_id) AS unique_customers
    FROM customer_lifecycle
""").show()

┌─────────────────┬──────────────────┐
│ total_customers │ unique_customers │
│      int64      │      int64       │
├─────────────────┼──────────────────┤
│         1362281 │          1362281 │
└─────────────────┴──────────────────┘



In [49]:
con.sql("""
DESCRIBE customer_lifecycle
""").show()

┌────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name   │ column_type │  null   │   key   │ default │  extra  │
│    varchar     │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ customer_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ first_date     │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ second_date    │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ days_to_second │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ eligible_90d   │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ ret_90d        │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



In [50]:
con.sql("""
    SELECT
        SUM(
        CASE
          WHEN eligible_90d = 1 THEN 1
          ELSE 0
          END ) AS eligible_customers,
        SUM(
        CASE 
          WHEN eligible_90d= 1 AND ret_90d = 1 THEN  1
          ELSE 0
          END) AS retained_customers,
         ROUND(SUM(
        CASE 
          WHEN eligible_90d= 1 AND ret_90d = 1 THEN  1
          ELSE 0
          END ) /SUM(
        CASE
          WHEN eligible_90d = 1 THEN 1
          ELSE 0
          END )  * 100.0 ,2) AS retention_90d
    FROM customer_lifecycle

""").show()
    
          

┌────────────────────┬────────────────────┬───────────────┐
│ eligible_customers │ retained_customers │ retention_90d │
│       int128       │       int128       │    double     │
├────────────────────┼────────────────────┼───────────────┤
│            1291147 │             607798 │         47.07 │
└────────────────────┴────────────────────┴───────────────┘



In [51]:
con.sql("""
    SELECT
        strftime(first_date, '%Y-%m') AS first_month,
        COUNT(eligible_90d) AS eligible_customers,
        SUM(ret_90d) AS retained_customers,
        ROUND(100.0 * AVG(ret_90d),2) AS retention_90d
    FROM customer_lifecycle
    WHERE eligible_90d = 1
    GROUP BY first_month
    ORDER BY first_month
   
    
""").show()


┌─────────────┬────────────────────┬────────────────────┬───────────────┐
│ first_month │ eligible_customers │ retained_customers │ retention_90d │
│   varchar   │       int64        │       int128       │    double     │
├─────────────┼────────────────────┼────────────────────┼───────────────┤
│ 2018-09     │             140340 │              97553 │         69.51 │
│ 2018-10     │             212669 │             133579 │         62.81 │
│ 2018-11     │             138233 │              72018 │          52.1 │
│ 2018-12     │              89944 │              41237 │         45.85 │
│ 2019-01     │              68957 │              29581 │          42.9 │
│ 2019-02     │              65453 │              25782 │         39.39 │
│ 2019-03     │              51185 │              22204 │         43.38 │
│ 2019-04     │              49884 │              21607 │         43.31 │
│ 2019-05     │              48645 │              20657 │         42.46 │
│ 2019-06     │              52127 │  

In [52]:
con.sql("""
    SELECT
        COUNT(eligible_90d) AS eligible_customers,
        SUM(ret_90d) AS retained_customers,
        ROUND(100.0 * AVG(ret_90d),2) AS retention_90d
    FROM customer_lifecycle
    WHERE eligible_90d= 1
   
   
    
""").show()

┌────────────────────┬────────────────────┬───────────────┐
│ eligible_customers │ retained_customers │ retention_90d │
│       int64        │       int128       │    double     │
├────────────────────┼────────────────────┼───────────────┤
│            1291147 │             607798 │         47.07 │
└────────────────────┴────────────────────┴───────────────┘



In [53]:
con.execute("""
    CREATE OR REPLACE TABLE mart_cohort_retention_90d AS
    SELECT
        strftime(first_date, '%Y-%m') AS first_month,
        COUNT(eligible_90d) AS eligible_customers,
        SUM(ret_90d) AS retained_customers,
        ROUND(100.0 * AVG(ret_90d),2) AS retention_90d
    FROM customer_lifecycle
    WHERE eligible_90d = 1
    GROUP BY first_month
    ORDER BY first_month
    
""")

In [54]:
con.sql("""
    SELECT
        COUNT(*) AS cohort_months,
        MIN(first_month) AS first_cohort,
        MAX(first_month) AS last_cohort,
        SUM(eligible_customers) AS total_eligible,
        SUM(retained_customers) AS total_retained
    FROM mart_cohort_retention_90d
""").show()

┌───────────────┬──────────────┬─────────────┬────────────────┬────────────────┐
│ cohort_months │ first_cohort │ last_cohort │ total_eligible │ total_retained │
│     int64     │   varchar    │   varchar   │     int128     │     int128     │
├───────────────┼──────────────┼─────────────┼────────────────┼────────────────┤
│            22 │ 2018-09      │ 2020-06     │        1291147 │         607798 │
└───────────────┴──────────────┴─────────────┴────────────────┴────────────────┘



In [55]:
con.execute("""
    CREATE OR REPLACE VIEW v_cohort_retention_90d AS
    SELECT *,
      CASE
       WHEN first_month < '2019-01' THEN 'start_boundary'
       WHEN first_month > '2020-05' THEN 'end_partial'
       ELSE 'main_analysis'
      END AS cohort_window
    FROM mart_cohort_retention_90d
""") 

In [56]:
con.sql("""
    SELECT
        cohort_window,
        COUNT(cohort_window) AS cohort_month
    FROM v_cohort_retention_90d
    GROUP BY cohort_window
    ORDER BY cohort_window
""").show()

┌────────────────┬──────────────┐
│ cohort_window  │ cohort_month │
│    varchar     │    int64     │
├────────────────┼──────────────┤
│ end_partial    │            1 │
│ main_analysis  │           17 │
│ start_boundary │            4 │
└────────────────┴──────────────┘



In [57]:
con.sql("""
    SELECT
        COUNT(*) AS cohort_months,
        SUM(eligible_customers) AS eligible_customers,
        SUM(retained_customers) AS retained_customers,
        ROUND( SUM(retained_customers)/SUM(eligible_customers) * 100.0 ,2) AS weighted_retention,
        MIN(retention_90d) AS min_monthly_retention,
        MAX(retention_90d) AS max_monthly_retention
    FROM v_cohort_retention_90d
    WHERE cohort_window = 'main_analysis'
""").show()

┌───────────────┬────────────────────┬────────────────────┬────────────────────┬───────────────────────┬───────────────────────┐
│ cohort_months │ eligible_customers │ retained_customers │ weighted_retention │ min_monthly_retention │ max_monthly_retention │
│     int64     │       int128       │       int128       │       double       │        double         │        double         │
├───────────────┼────────────────────┼────────────────────┼────────────────────┼───────────────────────┼───────────────────────┤
│            17 │             685335 │             254739 │              37.17 │                 29.78 │                 43.38 │
└───────────────┴────────────────────┴────────────────────┴────────────────────┴───────────────────────┴───────────────────────┘



## 90-Day Cohort Retention Analysis

Customers were grouped by the month of their first observed purchase. The 90-day retention rate measures the percentage of eligible customers who made a second observed purchase within 90 days. Customers without 90 complete days of follow-up were excluded from the denominator.

The complete eligible population produced a 90-day retention rate of 47.07%. The September–December 2018 cohorts were excluded from the main analysis to reduce sensitivity to the start of the observation window. This choice does not establish that later customers were genuinely new: purchases before the dataset began remain unknown. June 2020 was excluded because only first observed purchases through June 24 had a complete 90-day follow-up window.

For the main analysis, the comparable window was restricted to January 2019 through May 2020. Across these 17 monthly cohorts, 254,739 of 685,335 eligible customers made a second observed purchase within 90 days, resulting in a weighted retention rate of 37.17%.

Monthly retention ranged from 29.78% to 43.38%. The rate declined from 42.90% in January 2019 to 36.34% in May 2020, a decrease of 6.56 percentage points, although the pattern was not consistently downward in every month.

These results describe first-observed purchase cohorts rather than confirmed new-customer acquisition cohorts. The observed differences should not be interpreted as causal effects without further statistical analysis or experimental evidence.


In [58]:
con.sql("""
SELECT
    customer_id ,
    first_date,
    second_date,
    days_to_second,
    CASE
      WHEN days_to_second <= 30 THEN 1
      ELSE 0
    END AS ret_30d,
    CASE
      WHEN days_to_second <= 60 THEN 1
      ELSE 0
    END AS ret_60d
FROM customer_lifecycle
WHERE eligible_90d= 1
AND first_date >= DATE '2019-01-01'
AND first_date < DATE '2020-06-01'
ORDER BY days_to_second
LIMIT 20
""").show()

┌──────────────────────────────────────────────────────────────────┬────────────┬─────────────┬────────────────┬─────────┬─────────┐
│                           customer_id                            │ first_date │ second_date │ days_to_second │ ret_30d │ ret_60d │
│                             varchar                              │    date    │    date     │     int64      │  int32  │  int32  │
├──────────────────────────────────────────────────────────────────┼────────────┼─────────────┼────────────────┼─────────┼─────────┤
│ 96456adf674a1c8d720b67111d089f19911fc64169fd954548120fcd77b3c9e2 │ 2019-07-24 │ 2019-07-25  │              1 │       1 │       1 │
│ 966bb1046abf3b8a1e826c7153f8d770b3d48ab721a536ceaf5d461b9bfdbf36 │ 2019-01-20 │ 2019-01-21  │              1 │       1 │       1 │
│ 9731a0babd0861e52f603b7a0eb67a08e88c9ace5dd4d06b70d6e1bcf378ec92 │ 2019-11-15 │ 2019-11-16  │              1 │       1 │       1 │
│ 98881a42d9878ad800d17cc4964c94beaaf9c8b504fdcded83946410d56855d0 │ 

In [59]:
con.sql("""
SELECT DISTINCT
    days_to_second,
    ret_90d,
    CASE
      WHEN days_to_second <= 30 THEN 1
      ELSE 0
    END AS ret_30d,
    CASE
      WHEN days_to_second <= 60 THEN 1
      ELSE 0
    END AS ret_60d
FROM customer_lifecycle
WHERE days_to_second IN (29, 30, 31, 59, 60, 61, 89, 90, 91)
AND first_date >= DATE '2019-01-01'
AND first_date < DATE '2020-06-01'
ORDER BY days_to_second
LIMIT 20
""").show()

┌────────────────┬─────────┬─────────┬─────────┐
│ days_to_second │ ret_90d │ ret_30d │ ret_60d │
│     int64      │  int32  │  int32  │  int32  │
├────────────────┼─────────┼─────────┼─────────┤
│             29 │       1 │       1 │       1 │
│             30 │       1 │       1 │       1 │
│             31 │       1 │       0 │       1 │
│             59 │       1 │       0 │       1 │
│             60 │       1 │       0 │       1 │
│             61 │       1 │       0 │       0 │
│             89 │       1 │       0 │       0 │
│             90 │       1 │       0 │       0 │
│             91 │       0 │       0 │       0 │
└────────────────┴─────────┴─────────┴─────────┘



In [60]:
con.sql("""
WITH classified_customers AS (
SELECT
    days_to_second,
    customer_id,
    first_date,
    ret_90d,
    CASE
      WHEN days_to_second <= 30 THEN 1
      ELSE 0
    END AS ret_30d,
    CASE
      WHEN days_to_second <= 60 THEN 1
      ELSE 0
    END AS ret_60d
FROM customer_lifecycle
WHERE eligible_90d= 1
AND first_date >= DATE '2019-01-01'
AND first_date < DATE '2020-06-01'
)

SELECT 
    COUNT(*) AS eligible_customers,
    SUM(ret_30d) AS retained_30d,
    SUM(ret_60d) AS retained_60d,
    SUM(ret_90d) AS retained_90d,
    ROUND(
    100.0 * SUM(ret_30d) / COUNT(*),
    2) AS retention_30d,
    ROUND(
    100.0 * SUM(ret_60d) / COUNT(*),
    2 )AS retention_60d,
    ROUND(
    100.0 * SUM(ret_90d) / COUNT(*),
    2 )AS retention_90d
FROM classified_customers
ORDER BY retained_30d,
         retained_60d,
         retained_90d
""").show()


┌────────────────────┬──────────────┬──────────────┬──────────────┬───────────────┬───────────────┬───────────────┐
│ eligible_customers │ retained_30d │ retained_60d │ retained_90d │ retention_30d │ retention_60d │ retention_90d │
│       int64        │    int128    │    int128    │    int128    │    double     │    double     │    double     │
├────────────────────┼──────────────┼──────────────┼──────────────┼───────────────┼───────────────┼───────────────┤
│             685335 │       149800 │       210844 │       254739 │         21.86 │         30.77 │         37.17 │
└────────────────────┴──────────────┴──────────────┴──────────────┴───────────────┴───────────────┴───────────────┘



In [61]:
con.execute("""
WITH classified_customers AS (
SELECT
    days_to_second,
    customer_id,
    first_date,
    ret_90d,
    CASE
      WHEN days_to_second <= 30 THEN 1
      ELSE 0
    END AS ret_30d,
    CASE
      WHEN days_to_second <= 60 THEN 1
      ELSE 0
    END AS ret_60d
FROM customer_lifecycle
WHERE eligible_90d= 1
AND first_date >= DATE '2019-01-01'
AND first_date < DATE '2020-06-01'
)

SELECT 
    COUNT(*) AS eligible_customers,
    SUM(ret_30d) AS retained_30d,
    SUM(ret_60d) AS retained_60d,
    SUM(ret_90d) AS retained_90d,
    ROUND(
    100.0 * SUM(ret_30d) / COUNT(*),
    2) AS retention_30d,
    ROUND(
    100.0 * SUM(ret_60d) / COUNT(*),
    2 )AS retention_60d,
    ROUND(
    100.0 * SUM(ret_90d) / COUNT(*),
    2 )AS retention_90d
FROM classified_customers
ORDER BY retained_30d,
         retained_60d,
         retained_90d
""")

In [62]:
con.sql("""
    SELECT 
        COUNT(*) AS retained_customers,
        MEDIAN(days_to_second) AS median_days_to_second
    FROM customer_lifecycle
    WHERE eligible_90d = 1
    AND ret_90d = 1
    AND first_date >= DATE '2019-01-01'
    AND first_date < DATE '2020-06-01'
""").show()

┌────────────────────┬───────────────────────┐
│ retained_customers │ median_days_to_second │
│       int64        │        double         │
├────────────────────┼───────────────────────┤
│             254739 │                  22.0 │
└────────────────────┴───────────────────────┘



### Among customers in the main comparable cohort window who made a second observed purchase within 90 days, the median time to the second purchase was 22 days. Of all eligible customers, 21.86% returned within 30 days, 30.77% within 60 days, and 37.17% within 90 days. This indicates that repeat purchasing was concentrated in the first month, although a substantial share of 90-day repeat customers returned between days 31 and 90.

In [63]:
con.sql("""
WITH main_population AS (
SELECT
    days_to_second,
    customer_id,
    first_date,
    ret_90d,
    CASE
      WHEN days_to_second <= 30 THEN 1
      ELSE 0
    END AS ret_30d,
    CASE
      WHEN days_to_second <= 60 THEN 1
      ELSE 0
    END AS ret_60d
FROM customer_lifecycle
WHERE eligible_90d= 1
AND first_date >= DATE '2019-01-01'
AND first_date < DATE '2020-06-01'
)
SELECT
    COUNT(*) AS eligible_customers,
    SUM(ret_30d) AS retained_30d,
    ROUND(AVG(ret_30d)*100.0,2) AS ret_30d_pct,
    SUM(ret_60d) AS retained_60d,
    ROUND(AVG(ret_60d) * 100.0,2) AS ret_60d_pct,
    SUM(ret_90d) AS retained_90d,
    ROUND(AVG(ret_90d)* 100.0,2) AS ret_90d_pct,
    MEDIAN(
       CASE
         WHEN ret_90d =1 THEN days_to_second
         ELSE NULL
         END) AS median_days_to_second
FROM main_population 
""").show()

┌────────────────────┬──────────────┬─────────────┬──────────────┬─────────────┬──────────────┬─────────────┬───────────────────────┐
│ eligible_customers │ retained_30d │ ret_30d_pct │ retained_60d │ ret_60d_pct │ retained_90d │ ret_90d_pct │ median_days_to_second │
│       int64        │    int128    │   double    │    int128    │   double    │    int128    │   double    │        double         │
├────────────────────┼──────────────┼─────────────┼──────────────┼─────────────┼──────────────┼─────────────┼───────────────────────┤
│             685335 │       149800 │       21.86 │       210844 │       30.77 │       254739 │       37.17 │                  22.0 │
└────────────────────┴──────────────┴─────────────┴──────────────┴─────────────┴──────────────┴─────────────┴───────────────────────┘



### Retention velocity: 30, 60 and 90 days

### Retention velocity: 30, 60 and 90 days

The analysis includes 685,335 customers whose first observed purchase occurred between January 2019 and May 2020. The cumulative second-purchase rate increased from 21.86% within 30 days to 30.77% within 60 days and 37.17% within 90 days. Among customers who made a second observed purchase within 90 days, the median time to that purchase was 22 days. These results describe the timing and frequency of observed repeat purchases, but they do not establish what caused customers to return.


In [64]:
customers_relation = con.read_csv(str(data_dir / "customers.csv"))
customers_relation.create_view("customers", replace=True)

articles_relation = con.read_csv(str(data_dir / "articles.csv"))
articles_relation.create_view("articles", replace=True)

┌────────────┬──────────────┬───────────────────────────┬─────────────────┬───────────────────┬────────────────────┬─────────────────────────┬───────────────────────────┬───────────────────┬───────────────────┬───────────────────────────┬─────────────────────────────┬────────────────────────────┬──────────────────────────────┬───────────────┬───────────────────────┬────────────┬────────────────────────┬────────────────┬──────────────────┬────────────┬────────────────────────────────┬──────────────────┬────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ article_id │ product_code │         prod_name         │ product_type_no │ product_type_name │ product_group_name │ graphical_appearance_no │ graphical_appearance_name │ colour_group_code │ colour_group_name │ perceived_colour_

In [65]:
con.sql("""
    SELECT
        (SELECT COUNT(*) FROM customers) AS customer_rows,
        (SELECT COUNT(*) FROM articles) AS article_rows
""").show()

┌───────────────┬──────────────┐
│ customer_rows │ article_rows │
│     int64     │    int64     │
├───────────────┼──────────────┤
│       1371980 │       105542 │
└───────────────┴──────────────┘



In [66]:
con.sql("""
DESCRIBE customers
""").show()

┌────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│      column_name       │ column_type │  null   │   key   │ default │  extra  │
│        varchar         │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ customer_id            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ FN                     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ Active                 │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ club_member_status     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ fashion_news_frequency │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ age                    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ postal_code            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



In [67]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(customer_id) AS populated_ids,
    COUNT(DISTINCT customer_id) AS unique_customers,
    COUNT(*) - COUNT(customer_id) AS missing_ids,
    COUNT(customer_id) - COUNT(DISTINCT customer_id) AS duplicate_cust_rows
FROM customers
""").show()

┌────────────┬───────────────┬──────────────────┬─────────────┬─────────────────────┐
│ total_rows │ populated_ids │ unique_customers │ missing_ids │ duplicate_cust_rows │
│   int64    │     int64     │      int64       │    int64    │        int64        │
├────────────┼───────────────┼──────────────────┼─────────────┼─────────────────────┤
│    1371980 │       1371980 │          1371980 │           0 │                   0 │
└────────────┴───────────────┴──────────────────┴─────────────┴─────────────────────┘



In [68]:
con.sql("""
SELECT
    COUNT(*) - COUNT(age) AS missing_age,
    ROUND(
    100.0 * (COUNT(*) - COUNT(age)) / COUNT(*),
    2) AS missing_age_pct,
    COUNT(*) - COUNT( club_member_status) AS missing_status,
    ROUND(
    100.0 * (COUNT(*) - COUNT( club_member_status)) / COUNT(*),
    2) AS missing_status_pct,
    COUNT(*) - COUNT(fashion_news_frequency) AS missing_new_freq,
    ROUND(
    100.0 * (COUNT(*) - COUNT( fashion_news_frequency)) / COUNT(*),
    2) AS missing_new_freq_pct,
    COUNT(*) - COUNT(FN) AS missing_fn,
    COUNT(*) - COUNT(active) AS missing_active
FROM customers
""").show()
    

┌─────────────┬─────────────────┬────────────────┬────────────────────┬──────────────────┬──────────────────────┬────────────┬────────────────┐
│ missing_age │ missing_age_pct │ missing_status │ missing_status_pct │ missing_new_freq │ missing_new_freq_pct │ missing_fn │ missing_active │
│    int64    │     double      │     int64      │       double       │      int64       │        double        │   int64    │     int64      │
├─────────────┼─────────────────┼────────────────┼────────────────────┼──────────────────┼──────────────────────┼────────────┼────────────────┤
│       15861 │            1.16 │           6062 │               0.44 │            16009 │                 1.17 │     895050 │         907576 │
└─────────────┴─────────────────┴────────────────┴────────────────────┴──────────────────┴──────────────────────┴────────────┴────────────────┘



In [69]:
con.sql("""
SELECT
    FN,
    COUNT(*) AS customers
FROM customers
GROUP BY FN
ORDER BY FN DESC
""").show()

┌────────┬───────────┐
│   FN   │ customers │
│ double │   int64   │
├────────┼───────────┤
│    1.0 │    476930 │
│   NULL │    895050 │
└────────┴───────────┘



### Customer identifier and missing-value audit

The `customers` table contains 1,371,980 rows and the same number of unique, non-null customer IDs. Therefore, its observed grain is one row per customer, and `customer_id` can be used as the join key without creating duplicate customer records.

The `age`, `club_member_status`, and `fashion_news_frequency` columns have less than 1.2% missing values and can be considered for customer segmentation. In contrast, `FN` and `Active` have substantially higher missingness.

The `FN` column contains only `1.0` and `NULL`: 476,930 customers have the value `1.0`, while 895,050 have no recorded value. This suggests that `FN` is stored as a sparse binary indicator rather than a continuous numerical variable. However, `NULL` will be interpreted as “not indicated or not recorded,” rather than a confirmed negative value. Because the field may represent a customer snapshot rather than their status at the first observed purchase, any relationship with retention will be treated as descriptive and not causal.


# Inspect Active values to determine whether the column is encoded as a sparse binary indicator

In [70]:
con.sql("""
SELECT
    active,
    COUNT(*) AS customers
FROM customers
GROUP BY active
ORDER BY active DESC
""").show()

┌────────┬───────────┐
│ Active │ customers │
│ double │   int64   │
├────────┼───────────┤
│    1.0 │    464404 │
│   NULL │    907576 │
└────────┴───────────┘



The `Active` column follows the same sparse binary pattern: 464,404 customers are marked with `1.0`, while 907,576 have a null value. Because both `FN` and `Active` have limited coverage, ambiguous null values, and no historical timestamp, they will not be used as primary segmentation variables. The better-documented and more complete `fashion_news_frequency` field will be evaluated instead.

In [71]:
con.sql("""
SELECT
    fashion_news_frequency,
    COUNT(*) AS customers
FROM customers
GROUP BY fashion_news_frequency
ORDER BY fashion_news_frequency DESC
""").show()

┌────────────────────────┬───────────┐
│ fashion_news_frequency │ customers │
│        varchar         │   int64   │
├────────────────────────┼───────────┤
│ Regularly              │    477416 │
│ None                   │         2 │
│ NONE                   │    877711 │
│ Monthly                │       842 │
│ NULL                   │     16009 │
└────────────────────────┴───────────┘



# Standardize Fashion News categories into actionable CRM segments

In [72]:
con.sql("""
SELECT
    COUNT(*) AS customers,
    CASE 
      WHEN fashion_news_frequency IS  NULL THEN 'Unknown'
      WHEN UPPER(fashion_news_frequency) = 'NONE' THEN 'Not subscribed'
      WHEN fashion_news_frequency IN ('Regularly', 'Monthly')
      THEN 'Subscribed'
      ELSE 'Other'
      END AS news_segment
FROM customers
GROUP BY news_segment
ORDER BY customers DESC
""").show()

┌───────────┬────────────────┐
│ customers │  news_segment  │
│   int64   │    varchar     │
├───────────┼────────────────┤
│    877713 │ Not subscribed │
│    478258 │ Subscribed     │
│     16009 │ Unknown        │
└───────────┴────────────────┘



#### Fashion News frequency

The original `fashion_news_frequency` column contained inconsistent labels, including `NONE` and `None`, and a very small `Monthly` category. The values were standardized into three CRM-oriented segments: `Subscribed`, `Not subscribed`, and `Unknown`. The transformation preserved all 1,371,980 customer records. This variable will be used descriptively because the dataset does not indicate whether the recorded subscription status was already valid at the customer's first observed purchase.

In [73]:
con.sql("""
SELECT
    club_member_status,
    COUNT(*) AS customers
FROM customers
GROUP BY  club_member_status
ORDER BY  club_member_status DESC
""").show()

┌────────────────────┬───────────┐
│ club_member_status │ customers │
│      varchar       │   int64   │
├────────────────────┼───────────┤
│ PRE-CREATE         │     92960 │
│ LEFT CLUB          │       467 │
│ ACTIVE             │   1272491 │
│ NULL               │      6062 │
└────────────────────┴───────────┘



#### Club membership status

Most customers are classified as `ACTIVE` (92.75%), while `PRE-CREATE` represents 6.78%, `LEFT CLUB` only 0.03%, and 0.44% have an unknown status. For analysis, `PRE-CREATE` and `LEFT CLUB` will be combined into a broader `Not active member` segment, while null values will remain `Unknown`.

Because the table does not provide a timestamp for membership status, the recorded status may have been assigned after the first observed purchase. Therefore, any relationship between membership and retention will be interpreted as descriptive rather than causal.

# Inspect the age range and distribution before creating age segments

In [74]:
con.sql("""
SELECT
    MIN(age) AS min_age,
    QUANTILE_CONT(age, 0.25) AS p25_age,
    MEDIAN(age) AS median_age,
    QUANTILE_CONT(age, 0.75) AS p75_age,
    MAX(age) AS max_age
FROM customers
""").show()

┌─────────┬─────────┬────────────┬─────────┬─────────┐
│ min_age │ p25_age │ median_age │ p75_age │ max_age │
│  int64  │ double  │   double   │ double  │  int64  │
├─────────┼─────────┼────────────┼─────────┼─────────┤
│      16 │    24.0 │       32.0 │    49.0 │      99 │
└─────────┴─────────┴────────────┴─────────┴─────────┘



# Create interpretable age groups and validate their population sizes

In [75]:
con.sql("""
SELECT
    COUNT(*) AS customers,
    CASE
      WHEN age IS NULL THEN 'Unknown'
      WHEN age BETWEEN 16 AND 24 THEN '16-24'
      WHEN age BETWEEN 25 AND 34 THEN '25-34'
      WHEN age BETWEEN 35 AND 49 THEN '35-49'
      WHEN age >= 50 THEN '50+'
      ELSE 'Invalid'
    END AS age_group
FROM customers
GROUP BY age_group
ORDER BY customers DESC
""").show()

┌───────────┬───────────┐
│ customers │ age_group │
│   int64   │  varchar  │
├───────────┼───────────┤
│    393292 │ 25-34     │
│    357169 │ 16-24     │
│    317992 │ 50+       │
│    287666 │ 35-49     │
│     15861 │ Unknown   │
└───────────┴───────────┘



#### Customer age

Customer ages range from 16 to 99, with a median of 32. No implausible age values were identified. Age was grouped into four interpretable segments: `16-24`, `25-34`, `35-49`, and `50+`. Customers with missing age information were retained in an `Unknown` group. The transformation preserved all 1,371,980 customer records, and every valid age segment contains a sufficiently large population for comparison.

In [76]:
# Inspect the article schema and verify the join-key data type

con.sql("""
    DESCRIBE articles
""").show()

┌───────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name        │ column_type │  null   │   key   │ default │  extra  │
│          varchar          │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ article_id                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ product_code              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ prod_name                 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ product_type_no           │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ product_type_name         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ product_group_name        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ graphical_appearance_no   │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ graphical_appearance_name │ VARCHAR     │ YES     │ NULL    │ NULL    │ NU

In [77]:
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(article_id) AS populated_ids,
    COUNT(DISTINCT article_id) AS unique_articles,
    COUNT(*) - COUNT(article_id) AS missing_ids ,
    COUNT(article_id) - COUNT(DISTINCT article_id) AS duplicate_art_rows,
    COUNT(*) - COUNT(product_group_name ) AS missing_product,
    ROUND(
    100.0 * (COUNT(*) - COUNT(product_group_name)) / COUNT(*),
    2) AS missing_product_groups_pct

FROM articles
""").show()

┌────────────┬───────────────┬─────────────────┬─────────────┬────────────────────┬─────────────────┬────────────────────────────┐
│ total_rows │ populated_ids │ unique_articles │ missing_ids │ duplicate_art_rows │ missing_product │ missing_product_groups_pct │
│   int64    │     int64     │      int64      │    int64    │       int64        │      int64      │           double           │
├────────────┼───────────────┼─────────────────┼─────────────┼────────────────────┼─────────────────┼────────────────────────────┤
│     105542 │        105542 │          105542 │           0 │                  0 │               0 │                        0.0 │
└────────────┴───────────────┴─────────────────┴─────────────┴────────────────────┴─────────────────┴────────────────────────────┘



#### Articles table audit

The `articles` table contains 105,542 rows and the same number of unique, non-null article IDs. Therefore, its observed grain is one row per article, and `article_id` can be used as a join key. The selected `product_group_name` field has complete coverage. Both `articles.article_id` and `transactions.article_id` are stored as `VARCHAR`, so no data-type conversion is required before joining them.

# Validate article join coverage and identify unmatched transaction records

In [78]:
con.sql("""
SELECT
    COUNT(*) AS transactions_rows,
    SUM(
    CASE
        WHEN a.article_id IS NULL THEN 1
        ELSE 0
    END) AS unmatched_transaction_rows,
    COUNT(
    DISTINCT CASE
        WHEN a.article_id IS NULL THEN t.article_id
    END) AS unmatched_article_ids
FROM transactions AS t
LEFT JOIN articles AS a
ON t.article_id = a.article_id
""").show()


┌───────────────────┬────────────────────────────┬───────────────────────┐
│ transactions_rows │ unmatched_transaction_rows │ unmatched_article_ids │
│       int64       │           int128           │         int64         │
├───────────────────┼────────────────────────────┼───────────────────────┤
│          31788324 │                          0 │                     0 │
└───────────────────┴────────────────────────────┴───────────────────────┘



#### Article join coverage

All 31,788,324 transaction rows matched a valid record in the article catalogue. No unmatched article IDs were identified, resulting in 100% join coverage. Therefore, `product_group_name` can be added to transaction records without losing purchases due to missing catalogue information.

# Identify article lines belonging to each customer's first observed purchase date

In [79]:
con.sql("""
SELECT
    l.customer_id,
    l.first_date,
    COUNT(*) AS n_lines,
    COUNT(DISTINCT t.article_id) AS n_articles,
    COUNT(DISTINCT t.sales_channel_id) AS n_channels,
    COUNT(DISTINCT a.product_group_name) AS n_groups
FROM customer_lifecycle AS l
JOIN transactions AS t
ON l.customer_id = t.customer_id
LEFT JOIN articles AS a
ON t.article_id = a.article_id
WHERE eligible_90d= 1
AND first_date >= '2019-01-01'
AND first_date < '2020-06-01'
GROUP BY l.customer_id,
         l.first_date
LIMIT 20
""").show()

┌──────────────────────────────────────────────────────────────────┬────────────┬─────────┬────────────┬────────────┬──────────┐
│                           customer_id                            │ first_date │ n_lines │ n_articles │ n_channels │ n_groups │
│                             varchar                              │    date    │  int64  │   int64    │   int64    │  int64   │
├──────────────────────────────────────────────────────────────────┼────────────┼─────────┼────────────┼────────────┼──────────┤
│ e7a30e8e6905b6f6a90d236d47789414ec1070925939eb04c42b0d188d43ec47 │ 2019-07-25 │      36 │         32 │          2 │        7 │
│ eaf8c83f91dca23c9e0ee6dc1c294561cd486d4e787541773f90eacb5f1fb2d0 │ 2019-02-23 │      16 │         16 │          2 │        5 │
│ c8ecf25e092d08eec5dd42c486876107e16f0fff94c7b0a5b3e6a5b8b5853b2e │ 2020-05-16 │       4 │          4 │          1 │        3 │
│ 0672fd948a7a6f2d496f1e932ec4bc7d8a59111ffa0849536b7f6455456c146d │ 2019-01-06 │      85 │      

## First-purchase feature engineering

This section transforms first-purchase transaction lines into one analytical record per customer.

### First-purchase feature engineering

This section transforms the transaction lines recorded on each customer's first observed purchase date into customer-level features. These features will later be compared with 30-day, 60-day, and 90-day repeat-purchase outcomes.

In [80]:
con.sql("""
CREATE OR REPLACE TABLE first_purchase_features AS
SELECT
    l.customer_id,
    l.first_date,
    COUNT(*) AS n_lines,
    COUNT(DISTINCT t.article_id) AS n_articles,
    COUNT(DISTINCT t.sales_channel_id) AS n_channels,
    COUNT(DISTINCT a.product_group_name) AS n_groups,
    CASE
      WHEN COUNT(DISTINCT t.sales_channel_id) = 2 THEN 'Both channels'
      WHEN MIN(t.sales_channel_id) = 1 THEN 'Channel 1'
      ELSE 'Channel 2'
      END AS first_channel_group,
    CASE
      WHEN COUNT(DISTINCT a.product_group_name) > 1 THEN 'Multi-category'
      ELSE MIN(a.product_group_name)
      END AS first_category_group
FROM customer_lifecycle AS l
JOIN transactions AS t
ON l.customer_id = t.customer_id
AND l.first_date = t.t_dat
LEFT JOIN articles AS a
ON t.article_id = a.article_id
WHERE eligible_90d= 1
AND first_date >= '2019-01-01'
AND first_date < '2020-06-01'
GROUP BY l.customer_id,
         l.first_date

""")

In [81]:
con.sql("""
    SELECT
        customer_id,
        first_date,
        n_lines,
        n_articles
    FROM first_purchase_features
    WHERE  customer_id LIKE '00549656%'
""").show()

┌──────────────────────────────────────────────────────────────────┬────────────┬─────────┬────────────┐
│                           customer_id                            │ first_date │ n_lines │ n_articles │
│                             varchar                              │    date    │  int64  │   int64    │
├──────────────────────────────────────────────────────────────────┼────────────┼─────────┼────────────┤
│ 00549656241b634c8e0e6e5476b9b6ff1e3258b55c61c20634b6796b669191c5 │ 2019-01-01 │       5 │          3 │
└──────────────────────────────────────────────────────────────────┴────────────┴─────────┴────────────┘



In [82]:
con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT article_id) AS article
    FROM transactions
    WHERE customer_id LIKE '00549656241b634c8e0e6e5476b9b6ff1e3258b55c61c20634b6796b669191c5'
    AND t_dat = DATE '2019-01-01'
""").show()

┌────────────┬─────────┐
│ total_rows │ article │
│   int64    │  int64  │
├────────────┼─────────┤
│          5 │       3 │
└────────────┴─────────┘



In [83]:
con.sql("""
    SELECT
        f.customer_id,
        f.n_articles,
        f.first_channel_group,
        f.first_category_group,
        l.ret_90d
    FROM first_purchase_features AS f
    LEFT JOIN customer_lifecycle AS l
    ON f.customer_id= l.customer_id
    LIMIT 20
""").show()

┌──────────────────────────────────────────────────────────────────┬────────────┬─────────────────────┬──────────────────────┬─────────┐
│                           customer_id                            │ n_articles │ first_channel_group │ first_category_group │ ret_90d │
│                             varchar                              │   int64    │       varchar       │       varchar        │  int32  │
├──────────────────────────────────────────────────────────────────┼────────────┼─────────────────────┼──────────────────────┼─────────┤
│ 468a1ab24d3b50614f74247b300a49d60ec7e05540a8c4464f8a20d8e60cf7da │          1 │ Channel 2           │ Underwear            │       0 │
│ 468d503af46c1604113ebb5c9ca20adc5ac7c591391b860aab2df160bcc3d4db │          4 │ Channel 2           │ Garment Upper body   │       0 │
│ 4690b7c58ef37651eb48a217ffaf0a85de75a887f2793d465e06922d8b2cac52 │          1 │ Channel 1           │ Garment Upper body   │       1 │
│ 469a94d30d598ca4410453bd2871712148052fb

In [84]:
con.sql("""
    SELECT
        f.first_channel_group,
        COUNT(*) AS customers,
        SUM(l.ret_90d) AS retained,
        ROUND(100.0 * SUM(l.ret_90d) / COUNT(*), 2) AS ret_90d_pct,
        CASE
         WHEN f.n_articles = 1 THEN 'One article'
         ELSE 'Multiple articles'
        END AS article_segment
    FROM first_purchase_features AS f
    LEFT JOIN customer_lifecycle AS l
    ON f.customer_id= l.customer_id
    GROUP BY article_segment,
             f.first_channel_group
    LIMIT 20
""").show()

┌─────────────────────┬───────────┬──────────┬─────────────┬───────────────────┐
│ first_channel_group │ customers │ retained │ ret_90d_pct │  article_segment  │
│       varchar       │   int64   │  int128  │   double    │      varchar      │
├─────────────────────┼───────────┼──────────┼─────────────┼───────────────────┤
│ Channel 2           │    335639 │   126911 │       37.81 │ Multiple articles │
│ Channel 2           │    149870 │    45133 │       30.11 │ One article       │
│ Channel 1           │    137170 │    55871 │       40.73 │ Multiple articles │
│ Channel 1           │     60080 │    25263 │       42.05 │ One article       │
│ Both channels       │        38 │       23 │       60.53 │ One article       │
│ Both channels       │      2538 │     1538 │        60.6 │ Multiple articles │
└─────────────────────┴───────────┴──────────┴─────────────┴───────────────────┘



In [85]:
con.sql("""
    SELECT
        COUNT(*) AS rows_after_join,
        COUNT(DISTINCT f.customer_id) AS unique_customers,
        SUM(l.ret_90d) AS retained,
        COUNT(*) - COUNT(l.ret_90d) AS missing_retention,
        COUNT(c.customer_id) - COUNT(*) AS unmatched_customers,
        SUM(CASE
          WHEN c.customer_id IS NOT NULL AND c.age IS NULL THEN 1 ELSE 0 
        END) AS missing_age 
    FROM first_purchase_features AS f
    LEFT JOIN customer_lifecycle AS l
    ON f.customer_id= l.customer_id
    LEFT JOIN customers AS c
   ON f.customer_id = c.customer_id
    LIMIT 10
""").show()
        

┌─────────────────┬──────────────────┬──────────┬───────────────────┬─────────────────────┬─────────────┐
│ rows_after_join │ unique_customers │ retained │ missing_retention │ unmatched_customers │ missing_age │
│      int64      │      int64       │  int128  │       int64       │        int64        │   int128    │
├─────────────────┼──────────────────┼──────────┼───────────────────┼─────────────────────┼─────────────┤
│          685335 │           685335 │   254739 │                 0 │                   0 │        8505 │
└─────────────────┴──────────────────┴──────────┴───────────────────┴─────────────────────┴─────────────┘



In [86]:
con.execute("""
CREATE OR REPLACE TABLE customer_analysis_mart AS
  SELECT
      f.customer_id,
      f.n_articles,
      f.first_channel_group,
      f.first_date,
      l.ret_90d,
      c.age
 FROM first_purchase_features AS f
 LEFT JOIN customer_lifecycle AS l
 ON f.customer_id= l.customer_id
 LEFT JOIN customers AS c
 ON f.customer_id = c.customer_id
""")

In [87]:
analysis_df = con.sql("SELECT * FROM customer_analysis_mart").df()

In [88]:
analysis_df.shape

(685335, 6)

In [89]:
analysis_df.dtypes

customer_id                       str
n_articles                      int64
first_channel_group               str
first_date             datetime64[us]
ret_90d                         int32
age                             Int64
dtype: object

In [90]:
analysis_df['first_month'] = analysis_df["first_date"].dt.strftime("%Y-%m")

In [91]:
analysis_df[["first_date", "first_month"]].head()

,first_date,first_month
0,2019-12-06,2019-12
1,2020-04-12,2020-04
2,2019-11-30,2019-11
3,2019-12-01,2019-12
4,2019-01-02,2019-01


In [92]:
analysis_df["n_articles"].head(5) > 1

0    False
1     True
2     True
3     True
4    False
Name: n_articles, dtype: bool

In [93]:
analysis_df["multi_article"] = (
    analysis_df["n_articles"] > 1
).astype(int)

In [94]:
analysis_df["multi_article"].head(20)

0     0
1     1
2     1
3     1
4     0
5     1
6     1
7     0
8     0
9     0
10    1
11    1
12    0
13    1
14    1
15    1
16    1
17    0
18    1
19    1
Name: multi_article, dtype: int64

In [95]:
analysis_df[["n_articles", "multi_article", "ret_90d"]].head(5)

,n_articles,multi_article,ret_90d
0,1,0,0
1,3,1,0
2,3,1,0
3,2,1,1
4,1,0,1


In [96]:
analysis_df["age"].head(5) > 1

0    True
1    True
2    True
3    True
4    True
Name: age, dtype: boolean

In [97]:
analysis_df["missing_age"]=(
    analysis_df["age"].isna()
).astype(int)

In [98]:
analysis_df["missing_age"].sum()

np.int64(8505)

In [99]:
median_age = analysis_df["age"].median()

In [100]:
median_age

np.float64(31.0)

In [101]:
median_age = analysis_df["age"].median()
analysis_df["age_model"] = analysis_df["age"].fillna(median_age)

In [102]:
analysis_df.loc[
    analysis_df["missing_age"] == 1,
    ["age", "missing_age", "age_model"]
].head()

,age,missing_age,age_model
12,<NA>,1,31
176,<NA>,1,31
190,<NA>,1,31
237,<NA>,1,31
333,<NA>,1,31


In [103]:
analysis_df.columns.tolist()

['customer_id',
 'n_articles',
 'first_channel_group',
 'first_date',
 'ret_90d',
 'age',
 'first_month',
 'multi_article',
 'missing_age',
 'age_model']

In [104]:
analysis_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 685335 entries, 0 to 685334
Data columns (total 10 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   customer_id          685335 non-null  str           
 1   n_articles           685335 non-null  int64         
 2   first_channel_group  685335 non-null  str           
 3   first_date           685335 non-null  datetime64[us]
 4   ret_90d              685335 non-null  int32         
 5   age                  676830 non-null  Int64         
 6   first_month          685335 non-null  str           
 7   multi_article        685335 non-null  int64         
 8   missing_age          685335 non-null  int64         
 9   age_model            685335 non-null  Int64         
dtypes: Int64(2), datetime64[us](1), int32(1), int64(3), str(3)
memory usage: 51.0 MB


In [105]:
analysis_df.isna().sum()

customer_id               0
n_articles                0
first_channel_group       0
first_date                0
ret_90d                   0
age                    8505
first_month               0
multi_article             0
missing_age               0
age_model                 0
dtype: int64

In [106]:
analysis_df["ret_90d"].value_counts(dropna=False)

ret_90d
0    430596
1    254739
Name: count, dtype: int64

In [107]:
analysis_df["first_channel_group"].value_counts(dropna=False)

first_channel_group
Channel 2        485509
Channel 1        197250
Both channels      2576
Name: count, dtype: int64

In [108]:
analysis_df[["ret_90d", "multi_article"]].value_counts().sort_index()

ret_90d  multi_article
0        0                139569
         1                291027
1        0                 70419
         1                184320
Name: count, dtype: int64

In [109]:
analysis_df.groupby("multi_article")["ret_90d"].mean()

multi_article
0    0.335348
1    0.387759
Name: ret_90d, dtype: float64

In [111]:
%pip install statsmodels
import statsmodels.formula.api as smf



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [112]:
formula_unadjusted = "ret_90d ~ multi_article"

model_unadjusted = smf.logit(
    formula=formula_unadjusted,
    data=analysis_df
)

result_unadjusted = model_unadjusted.fit()

Optimization terminated successfully.
         Current function value: 0.658594
         Iterations 4


In [113]:
result_unadjusted.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                ret_90d   No. Observations:               685335
Model:                          Logit   Df Residuals:                   685333
Method:                           MLE   Df Model:                            1
Date:                Sun, 06 Sep 2026   Pseudo R-squ.:                0.001910
Time:                        17:18:57   Log-Likelihood:            -4.5136e+05
converged:                       True   LL-Null:                   -4.5222e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
=================================================================================
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept        -0.6841      0.005   -147.999      0.000      -0.693      -0.675
multi_article     0.2274      0.005     41.353      0.000       0.217       0.238
=================================================================================
"""

In [117]:
formula_2 = """
ret_90d ~ multi_article
       + age_model
       + missing_age
"""
model_2= smf.logit(
    formula=formula_2,
    data= analysis_df
)
result_2=model_2.fit() 

Optimization terminated successfully.
         Current function value: 0.657588
         Iterations 5


In [118]:
result_2.params

Intercept       -0.526411
multi_article    0.222683
age_model       -0.004079
missing_age     -0.724615
dtype: float64

In [121]:
formula_3 = """
ret_90d ~ multi_article
        + age_model
        + missing_age
        + C(first_channel_group)
"""
model_3 = smf.logit(
         formula = formula_3,
         data = analysis_df
         )
result_3 = model_3.fit()
result_3.params

Optimization terminated successfully.
         Current function value: 0.655787
         Iterations 5


Intercept                              0.379747
C(first_channel_group)[T.Channel 1]   -0.720572
C(first_channel_group)[T.Channel 2]   -0.964793
multi_article                          0.217540
age_model                             -0.004458
missing_age                           -0.697343
dtype: float64

In [122]:
analysis_df.groupby("first_channel_group")["ret_90d"].agg(
    customers="size",
    retention="mean"
)

,customers,retention
first_channel_group,,
Both channels,2576,0.605978
Channel 1,197250,0.411326
Channel 2,485509,0.354358


In [123]:
formula_4 = """
ret_90d ~ multi_article
        + age_model
        + missing_age
        + C(first_channel_group)
        + C(first_month)
"""

model_4 = smf.logit(
    formula=formula_4,
    data=analysis_df
)

result_4 = model_4.fit()

result_4.params

Optimization terminated successfully.
         Current function value: 0.651354
         Iterations 5


Intercept                              0.631891
C(first_channel_group)[T.Channel 1]   -0.720333
C(first_channel_group)[T.Channel 2]   -0.922156
C(first_month)[T.2019-02]             -0.177984
C(first_month)[T.2019-03]              0.013237
C(first_month)[T.2019-04]              0.009810
C(first_month)[T.2019-05]             -0.029530
C(first_month)[T.2019-06]             -0.195142
C(first_month)[T.2019-07]             -0.346331
C(first_month)[T.2019-08]             -0.342538
C(first_month)[T.2019-09]             -0.289526
C(first_month)[T.2019-10]             -0.414309
C(first_month)[T.2019-11]             -0.492693
C(first_month)[T.2019-12]             -0.555804
C(first_month)[T.2020-01]             -0.519279
C(first_month)[T.2020-02]             -0.577256
C(first_month)[T.2020-03]             -0.412529
C(first_month)[T.2020-04]             -0.410807
C(first_month)[T.2020-05]             -0.270836
multi_article                          0.193175
age_model                             -0

In [124]:
result_4.pvalues["multi_article"]

np.float64(2.1621928772726793e-263)

In [125]:
result_4.conf_int().loc["multi_article"]

0    0.182255
1    0.204095
Name: multi_article, dtype: float64

In [126]:
df_one = analysis_df.copy()
df_one["multi_article"] = 0

df_multi = analysis_df.copy()
df_multi["multi_article"] = 1

pred_one = result_4.predict(df_one)
pred_multi = result_4.predict(df_multi)

print("1 article:", pred_one.mean())
print("Multiple articles:", pred_multi.mean())
print("Difference:", pred_multi.mean() - pred_one.mean())

1 article: 0.3410845651067219
Multiple articles: 0.385074663123092
Difference: 0.043990098016370105


In [128]:
channel_month = (
    analysis_df[
        analysis_df["first_channel_group"].isin(["Channel 1", "Channel 2"])
    ]
    .groupby(["first_month", "first_channel_group"])
    .agg(
        customers=("customer_id", "size"),
        retained=("ret_90d", "sum"),
        retention_90d=("ret_90d", "mean")
    )
    .reset_index()
)

channel_month

,first_month,first_channel_group,customers,retained,retention_90d
0,2019-01,Channel 1,22155,9894,0.446581
1,2019-01,Channel 2,46526,19505,0.419228
2,2019-02,Channel 1,31975,12069,0.377451
3,2019-02,Channel 2,33161,13521,0.407738
4,2019-03,Channel 1,18970,8507,0.448445
5,2019-03,Channel 2,31989,13558,0.423833
6,2019-04,Channel 1,16806,8225,0.489409
7,2019-04,Channel 2,32808,13218,0.402890
8,2019-05,Channel 1,15816,7877,0.498040
9,2019-05,Channel 2,32567,12603,0.386987


In [129]:
channel_retention = channel_month.pivot(
    index="first_month",
    columns="first_channel_group",
    values="retention_90d"
)

In [130]:
channel_retention["gap_pp"] = (
    channel_retention["Channel 1"]
    - channel_retention["Channel 2"]
) * 100

channel_retention.round(3)

first_channel_group,Channel 1,Channel 2,gap_pp
first_month,,,
2019-01,0.447,0.419,2.735
2019-02,0.377,0.408,-3.029
2019-03,0.448,0.424,2.461
2019-04,0.489,0.403,8.652
2019-05,0.498,0.387,11.105
2019-06,0.477,0.352,12.509
2019-07,0.429,0.322,10.763
2019-08,0.397,0.328,6.888
2019-09,0.401,0.339,6.221


In [131]:
channel_counts = channel_month.pivot(
    index="first_month",
    columns="first_channel_group",
    values="customers"
)

channel_counts

first_channel_group,Channel 1,Channel 2
first_month,,
2019-01,22155.0,46526.0
2019-02,31975.0,33161.0
2019-03,18970.0,31989.0
2019-04,16806.0,32808.0
2019-05,15816.0,32567.0
2019-06,12844.0,39051.0
2019-07,11002.0,32196.0
2019-08,7765.0,20027.0
2019-09,8126.0,20723.0
